# Лабораторная работа №7
## Стекинг, многослойный персептрон и МГУА (GMDH)

**Постановка (по заданию):** датасет, предобработка, `train_test_split`, модели — **стекинг**, **MLP**, по желанию **МГУА** (линейный **Combi** + нелинейный **Mia**), **одна** метрика, сравнение.

> **Важно:** библиотека `gmdh` предназначена для **регрессии**; классификация в задании явно не поддерживается. Поэтому используется задача **регрессии** (California Housing).


## 1. Импорт данных и предобработка

Набор **California Housing** ( sklearn ): только числовые признаки. Стандартизация (`StandardScaler`) нужна в первую очередь для MLP и для сопоставимого масштаба при стекинге; те же матрицы подаём в МГУА.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name="MedHouseVal")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Размер train / test:", X_train_s.shape, X_test_s.shape)
print("Пропуски в X_train:", np.isnan(X_train_s).sum())


## 2. Модель стекинга (`StackingRegressor`)


In [ ]:
base_estimators = [
    ("rf", RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1)),
    ("gbr", GradientBoostingRegressor(random_state=RANDOM_STATE)),
]

stack_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 25)),
    cv=5,
    n_jobs=-1,
)

stack_model.fit(X_train_s, y_train)
pred_stack = stack_model.predict(X_test_s)
r2_stack = r2_score(y_test, pred_stack)
print("Stacking R^2 на тесте:", round(r2_stack, 6))


## 3. Многослойный персептрон (`MLPRegressor`)


In [ ]:
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=800,
    early_stopping=True,
    random_state=RANDOM_STATE,
    validation_fraction=0.1,
)

mlp.fit(X_train_s, y_train)
pred_mlp = mlp.predict(X_test_s)
r2_mlp = r2_score(y_test, pred_mlp)
print("MLP R^2 на тесте:", round(r2_mlp, 6))


## 4. МГУА: линейный **Combi** и нелинейный **Mia**

Установка: `pip install gmdh` (нужны **Boost** и компилятор C++; при ошибке CMake/Boost — см. блок обратной связи в конце).

Если импорт не удался, ячейка ниже помечает метрики как недоступные и не прерывает ноутбук.


In [ ]:
GMDH_OK = False
r2_combi = np.nan
r2_mia = np.nan
pred_combi = pred_mia = None

try:
    import gmdh
    from gmdh import Combi, Mia, Criterion, CriterionType

    GMDH_OK = True
except Exception as e:
    print("Библиотека gmdh недоступна:", repr(e))
    print("Типичная причина: нет Boost (linux: пакет boost-devel) или ошибка сборки wheel.")

if GMDH_OK:
    crit = gmdh.Criterion(criterion_type=CriterionType.REGULARITY)

    combi = Combi()
    combi.fit(
        X_train_s.astype(np.float64),
        y_train.to_numpy(dtype=np.float64),
        criterion=crit,
        test_size=0.25,
        n_jobs=-1,
        verbose=0,
    )
    pred_combi = combi.predict(X_test_s.astype(np.float64))
    r2_combi = r2_score(y_test, pred_combi)

    mia = Mia()
    mia.fit(
        X_train_s.astype(np.float64),
        y_train.to_numpy(dtype=np.float64),
        criterion=crit,
        k_best=3,
        polynomial_type=gmdh.PolynomialType.QUADRATIC,
        test_size=0.25,
        n_jobs=-1,
        verbose=0,
    )
    pred_mia = mia.predict(X_test_s.astype(np.float64))
    r2_mia = r2_score(y_test, pred_mia)

    print("Combi (линейный, комбинаторный) R^2:", round(r2_combi, 6))
    print("Mia (нелинейный, многослойный) R^2:", round(r2_mia, 6))
else:
    print("Пропуск обучения GMDH — установите зависимости и перезапустите ячейку.")


## 5. Сравнение по одной метрике — **R²** (коэффициент детерминации на тесте)


In [ ]:
rows = [
    {"Модель": "StackingRegressor (RF+GBR → RidgeCV)", "R2_test": r2_stack},
    {"Модель": "MLPRegressor (64-32)", "R2_test": r2_mlp},
    {"Модель": "GMDH Combi (линейный)", "R2_test": r2_combi},
    {"Модель": "GMDH Mia (нелинейный)", "R2_test": r2_mia},
]
res = pd.DataFrame(rows).sort_values("R2_test", ascending=False)
display(res)

fig, ax = plt.subplots(figsize=(9, 4))
plot_df = res.dropna(subset=["R2_test"])
ax.barh(plot_df["Модель"], plot_df["R2_test"], color=["#3a7ca5", "#f4a261", "#2a9d8f", "#e76f51"][: len(plot_df)])
ax.set_xlabel("$R^2$ на тесте")
ax.set_title("Сравнение моделей (ЛР 7)")
ax.axvline(0, color="gray", lw=0.8)
plt.tight_layout()
plt.show()


## 6. Выводы

- Задача **регрессии** выбрана из‑за ограничений `gmdh` по классификации.
- Проведены масштабирование признаков и разбиение `train_test_split`.
- Сравнение по **одной** метрике — **$R^2$** на тестовой выборке.
- Если `gmdh` не установился — оформите об этом обратную связь в канале (см. ниже) и приложите скрин CMake/Boost.


## 7. Обратная связь по библиотеке `gmdh` (канал ИУ5, тема **ТМО_МГУА**)

Краткий чеклист для поста (по заданию):

1. **Ошибки / баги** при `pip install gmdh` или при `import` (приложить скрин полного текста; за баг +1 балл на экзамене по формулировке методички).
2. **Опечатки** в документации (например, в тексте `gmdh`: *«Specitying»* → *Specifying* в `PolynomialType`).
3. **Трудности установки:** необходимость Boost ≥ 1.79, CMake, компилятор; отсутствие готового wheel под вашу ОС/Python.
4. **Предложения:** публикация manylinux-wheel, указание зависимостей в README.

Пример темы ошибки сборки: *CMake Error: Could NOT find Boost (...)* — скрин консоли + версия ОС/Python.
